[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_3_camera_model/08_distortion_undistortion.ipynb)

# Notebook 08 — Distortion and Undistortion
### 6D Pose Vision Workshop · Part 3: Camera Model

**Estimated time:** 40 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video notes 3 (*Camera Calibration Explained*), 4 (*OpenCV Python Camera Calibration*)

---

## Recommended Watch

> These are the same videos recommended for NB 07 — they cover distortion and undistortion as well. Skip if you already watched them.

| # | Title | Link | Duration |
|---|---|---|---|
| 1 | **Camera Calibration in less than 5 Minutes with OpenCV** | [▶ Watch](https://www.youtube.com/watch?v=_-BTKiamRTg) | ~5 min |
| 2 | **OpenCV Python Camera Calibration (Intrinsic, Extrinsic, Distortion)** | [▶ Watch](https://www.youtube.com/watch?v=H5qbRTikxI4) | ~20 min |

Watch this one **before Section 1** — it explains *why* lenses produce distortion physically (compound lens design, vignetting, chromatic aberration, barrel and tangential distortion) so the math in Section 1 has context:

| # | Title | Link |
|---|---|---|
| 3 | **Lens Related Issues \| Image Formation** *(VN36)* | [▶ Watch](https://youtu.be/hzOeqCb2Fg4?si=x0rpAcLTLkfNpT1p) |

---

## Optional Deep Dive

> Not required for this notebook — watch if you want a stronger theoretical foundation on how images are formed and sensed.

| # | Title | Link |
|---|---|---|
| 4 | **Image Formation, Image Sensing & Binary Images** *(playlist)* | [▶ Playlist](https://youtube.com/playlist?list=PL2zRqk16wsdr9X5rgF-d0pkzPdkHZ4KiT) |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os, json

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(5, 4)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

In [ ]:
# Load calibration from previous notebook (or use defaults)
calib_path = '../assets/calibration/camera_calibration.npz'

# CALIB_SOURCE persists through the session — later cells use it to label output
# Values: 'real' (NB07 data loaded) | 'default' (fallback / demo mode)
CALIB_SOURCE = 'default'

if os.path.exists(calib_path):
    data = np.load(calib_path)
    K    = data['K']
    dist = data['dist']
    CALIB_SOURCE = 'real'
    print(f"✓  Calibration loaded from  {calib_path}")
else:
    K    = np.array([[720, 0, 320], [0, 720, 240], [0, 0, 1]], dtype=np.float64)
    dist = np.array([-0.3, 0.1, 0.001, 0.002, -0.05])

    if IN_COLAB:
        print("⚠  Calibration file not found — running in DEMO MODE.")
        print()
        print("   In Colab each notebook has its own runtime; files written by NB07")
        print("   do not automatically appear here. Fix with one of these:")
        print()
        print("   Option 1 — run NB07 first in THIS same Colab session.")
        print("              (Files persist within a session; re-run this cell after.)")
        print()
        print("   Option 2 — mount Google Drive in both notebooks and point")
        print("              calib_path to the Drive path where NB07 saved the file.")
        print()
        print("   Continuing with built-in default values (demo mode).")
    else:
        print("⚠  Calibration file not found.")
        print("   Run NB07 first — it saves camera_calibration.npz — then re-run this cell.")
        print("   Continuing with built-in default values (demo mode).")

print()
print(f"── CALIB_SOURCE = '{CALIB_SOURCE}' ──────────────────────────────────────────")
print(f"K:\n{K}")
print(f"dist: {dist.flatten()}")

## Where you left off in NB07 — and what happens next

At the end of NB07 you ran `cv2.calibrateCamera` and got two things:

```python
K    = [[fx,  0, cx],   # intrinsic matrix — maps camera coords to pixels
        [ 0, fy, cy],
        [ 0,  0,  1]]

dist = [k1, k2, p1, p2, k3]   # distortion coefficients — how your lens bends light
```

The code cell above loads those results from the `.npz` file NB07 saved. If you see "Using default calibration values", go back and run NB07 first.

**What NB07 told us:** the camera has distortion. The `dist` values are non-zero.

**What NB08 does:** actually *fix* it — remove the distortion from images so that straight lines in the world look straight in the image.

---

### What's recap and what's new in this notebook

> **Section 1 — Types of Lens Distortion** → **recap from NB07**. You already saw radial and tangential distortion explained in NB07 Section 3. This section repeats it briefly with visualizations. Skim it to refresh the concept, or skip straight to Section 2 if it's still fresh.

> **Section 2 onwards → new content.** This is where you learn the actual tools: `cv2.undistort`, `cv2.remap`, `getOptimalNewCameraMatrix`, the `alpha` parameter, and the real-time undistortion loop. None of this was in NB07.

---

## Learning Objectives

By the end of this notebook you will:

- Understand what **radial** and **tangential** distortion are physically
- Know the **5 distortion coefficients** `[k1, k2, p1, p2, k3]` and what each affects
- Apply `cv2.undistort` for simple one-shot undistortion
- Use `cv2.remap` for **real-time** undistortion (precompute maps, then apply per frame)
- Know when to use `alpha=0` vs `alpha=1` in `getOptimalNewCameraMatrix`
- Validate undistortion results visually

---

## 1. Types of Lens Distortion *(recap from NB07)*

> This section reviews what you already learned in NB07 Section 3 — same concepts, now with interactive visualizations so you can see the effect. If the theory is still fresh, skim the diagrams and move on to Section 2.

### Why real cameras distort

The pinhole camera model assumes a perfect lens — straight rays, no bending. Real lenses bend light, and this bending increases the further you get from the optical axis (center). This creates **distortion**.

### Radial distortion — the most common type

Points are displaced radially outward or inward from the image center:

$$x_{\text{distorted}} = x(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$
$$y_{\text{distorted}} = y(1 + k_1 r^2 + k_2 r^4 + k_3 r^6)$$

Where $r^2 = x^2 + y^2$ is the squared distance from the image center.

**Annotation:**
- $k_1, k_2, k_3$ — radial distortion coefficients
- $r$ — normalized distance from center
- Negative $k_1$ → barrel distortion (pincushion if positive)

```
Barrel distortion (k1 < 0)       Pincushion distortion (k1 > 0)
  ┌───────────────┐               ┌──────────────────────────────┐
  │ ╔═══════════╗ │               │ ╔════╗                ╔════╗ │
  │ ║  bulges   ║ │               │ ║    ║                ║    ║ │
  │ ║  outward  ║ │               │ ║    ║    pinches     ║    ║ │
  │ ╚═══════════╝ │               │ ╚════╝    inward      ╚════╝ │
  └───────────────┘               └──────────────────────────────┘
  (wide-angle, fisheye lenses)    (telephoto, budget webcams)
```

### Tangential distortion — lens misalignment

Caused by the lens not being perfectly parallel to the image sensor:

$$x_{\text{distorted}} = x + [2p_1 xy + p_2(r^2 + 2x^2)]$$
$$y_{\text{distorted}} = y + [p_1(r^2 + 2y^2) + 2p_2 xy]$$

Where $p_1, p_2$ are the tangential distortion coefficients. Usually very small in modern cameras.

### OpenCV's distortion coefficient vector

$$\mathbf{d} = [k_1, \; k_2, \; p_1, \; p_2, \; k_3]$$

This is the `dist` array returned by `calibrateCamera`. For most cameras, $|k_3|$ and $|p_1|, |p_2|$ are very small.

In [ ]:
# ── Utility functions (used throughout this notebook) ────────────────────────

def simulate_distortion_for_viz(image, k1, k2=0, k3=0, K_mat=None):
    """[VIZ TRICK] Approximates distortion by applying the undistort map to a
    clean image.  Quick to compute and good enough for illustration.
    DO NOT use this for round-trip tests — undistorting the result will NOT
    recover the original (use make_distorted_image for that instead).
    """
    h, w = image.shape[:2]
    if K_mat is None:
        K_mat = np.array([[w*0.8, 0, w/2], [0, w*0.8, h/2], [0, 0, 1]], dtype=np.float64)
    dist_coeffs = np.array([k1, k2, 0, 0, k3])
    map1, map2 = cv2.initUndistortRectifyMap(K_mat, dist_coeffs, None, K_mat, (w, h), cv2.CV_32FC1)
    return cv2.remap(image, map1, map2, cv2.INTER_LINEAR)


def make_distorted_image(clean, K_mat, dist_coeffs, pad=150):
    """[TRUE MODEL] Create a properly distorted image via cv2.undistortPoints.

    For each pixel (xd, yd) in the distorted output, undistortPoints finds
    the corresponding pixel (xu, yu) in the clean source:

        distorted[yd, xd] = clean[yu, xu]

    This is exactly what a real lens captures.  Undistorting the result with
    initUndistortRectifyMap + remap recovers the original pixel-perfectly.

    The pad parameter extends the clean image with reflected content before
    remapping.  This prevents black corners: with barrel distortion the corners
    of the distorted image map slightly outside the source bounds — on a real
    camera sensor those pixels exist; in our synthetic grid they would be black
    without padding.
    """
    h, w = clean.shape[:2]
    xd, yd = np.meshgrid(np.arange(w, dtype=np.float32),
                         np.arange(h, dtype=np.float32))
    pts = np.stack([xd.ravel(), yd.ravel()], axis=1).reshape(-1, 1, 2)
    pts_und = cv2.undistortPoints(pts, K_mat, dist_coeffs, P=K_mat)
    pts_und = pts_und.reshape(h, w, 2)
    fwd_x, fwd_y = pts_und[:, :, 0], pts_und[:, :, 1]

    padded = cv2.copyMakeBorder(clean, pad, pad, pad, pad, cv2.BORDER_REFLECT)
    return cv2.remap(padded, fwd_x + pad, fwd_y + pad, cv2.INTER_LINEAR)


def create_grid_image(h=400, w=400, spacing=40):
    """Clean grid image — straight lines + colored intersection nodes.

    Every grid crossing gets a colored dot. Color cycles by (col+row) index
    so each node has a distinct color, making it easy to track individual
    points through distortion and recovery visually.
    """
    img = np.ones((h, w, 3), dtype=np.uint8) * 240
    for x in range(0, w, spacing):
        cv2.line(img, (x, 0), (x, h), (150, 150, 150), 1)
    for y in range(0, h, spacing):
        cv2.line(img, (0, y), (w, y), (150, 150, 150), 1)
    # 6 distinct colors — easy to distinguish, cycles diagonally across grid
    colors = [
        ( 30,  80, 220),   # blue
        (220,  50,  50),   # red
        ( 40, 180,  60),   # green
        (220, 140,  20),   # orange
        (160,  40, 200),   # purple
        ( 20, 180, 180),   # cyan
    ]
    for ci, x in enumerate(range(spacing, w, spacing)):
        for ri, y in enumerate(range(spacing, h, spacing)):
            color = colors[(ci + ri) % len(colors)]
            cv2.circle(img, (x, y), 6, color, -1)
            cv2.circle(img, (x, y), 6, (60, 60, 60), 1)   # thin dark outline
    return img


# ── Section 1: visualize what each coefficient looks like ────────────────────
# These use simulate_distortion_for_viz (VIZ TRICK) — the goal here is only
# to show what barrel vs pincushion vs fisheye looks like qualitatively.
# Watch how the colored nodes shift position with each distortion type.
grid  = create_grid_image()
K_vis = np.array([[320, 0, 200], [0, 320, 200], [0, 0, 1]], dtype=np.float64)

cases = [
    (grid,                                                             'Original\n(no distortion)'),
    (simulate_distortion_for_viz(grid, -0.5, 0.2, 0, K_vis), '[VIZ TRICK] k1=-0.5\nbarrel'),
    (simulate_distortion_for_viz(grid,  0.5, 0.0, 0, K_vis), '[VIZ TRICK] k1=+0.5\npincushion'),
    (simulate_distortion_for_viz(grid, -0.8, 0.5, 0, K_vis), '[VIZ TRICK] k1=-0.8, k2=+0.5\nfisheye-like'),
]

show_images(cases, figsize_per=(4, 4))
print("Each colored dot is one grid intersection — track how they shift with distortion.")
print()
print("[VIZ TRICK] = simulate_distortion_for_viz: visual approximation only.")
print("  Good for illustrating what each coefficient does.")
print("  NOT suitable for round-trip tests.")
print()
print("[TRUE MODEL] = make_distorted_image: uses cv2.undistortPoints — exact forward model.")
print("  Used in Sections 2 and 4 where undistortion must recover the original.")

## 2. Undistortion Methods ← new content starts here

You have K and `dist` from calibration. The goal now is simple: **remove the distortion from an image** so straight lines in the world look straight.

---

### Before anything: why does K change after undistortion?

When you warp a distorted frame back to a flat grid, some output pixels have no valid source data — they land outside the original sensor area. You get **black triangles at the corners**.

```
Distorted frame        Undistorted (alpha=1)        Undistorted (alpha=0)
┌──────────────┐       ┌──────────────┐             ┌───────────┐
│  ╔════════╗  │       │▓╔══════════╗▓│             │╔═════════╗│
│  ║ barrel ║  │  ──►  │▓║ straight ║▓│   or  ──►  │║ straight║│
│  ║  lines ║  │       │▓║  lines   ║▓│             │║  lines  ║│
│  ╚════════╝  │       │▓╚══════════╝▓│             │╚═════════╝│
└──────────────┘       └──────────────┘             └───────────┘
                        ▓ = black border            no black border, but
                          (no valid data)           slightly tighter FOV
```

`getOptimalNewCameraMatrix` computes a **new K** for the undistorted image and an `alpha` that controls this trade-off:

```python
K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
```

| alpha | What you get |
|-------|-------------|
| `0` | Tight crop — **no black borders**, lose a few pixels at the edges |
| `1` | Full field of view — **black triangles** appear at corners |
| `0 → 1` | Smooth blend between the two |

> **For robotics / pose estimation: always use `alpha=0`.** Black corner regions have no image data and confuse feature detectors (ArUco, SIFT) and `solvePnP`.

---

### Two ways to undistort — same result, very different speed

- **`cv2.undistort`** — one call. Simple. Recomputes the pixel mapping *internally on every call*.  
  Use this for offline processing of single images.

- **`cv2.remap`** — split into two steps: precompute the mapping *once*, then apply it to every frame.  
  **3–5× faster per frame.** Use this for any real-time video loop.

---

### Method A — cv2.undistort (simple, offline use)

```python
K_new, roi  = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha=0)
undistorted = cv2.undistort(img, K, dist, None, K_new)
```

### Method B — cv2.remap (fast, real-time use)

```python
# ── Do this ONCE, before your video loop ──
K_new, roi    = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha=0)
mapx, mapy    = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)

# ── Do this every frame ──
undistorted   = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
```

`mapx` / `mapy` are lookup tables: for every output pixel they store "go sample from this location in the distorted input". Building them is slow; applying them is fast.

In [ ]:
# Goal: show that Method A (cv2.undistort) and Method B (cv2.remap) both
# correctly remove barrel distortion — lines must be STRAIGHT in panels 3 and 4.
#
# [TRUE MODEL] is used here so undistortion cancels exactly.
# alpha=1 → black corners visible.  alpha=0 → tight crop, no black corners.

h, w = 480, 640
clean = create_grid_image(h, w, spacing=50)

# Camera model for this demo — strong barrel, like a cheap wide-angle webcam
K_strong    = np.array([[600, 0, w/2], [0, 600, h/2], [0, 0, 1]], dtype=np.float64)
dist_strong = np.array([-0.4, 0.15, 0.001, 0.002, -0.05])

# ── [TRUE MODEL] distorted frame ──────────────────────────────────────────────
# make_distorted_image pads the source grid before remapping so barrel
# distortion corners find grid content instead of black.
distorted = make_distorted_image(clean, K_strong, dist_strong)

# ── Method A: cv2.undistort ───────────────────────────────────────────────────
# alpha=1 → keep full field of view  →  black triangles appear at corners
K_new_a, roi_a = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=1)
undist_a = cv2.undistort(distorted, K_strong, dist_strong, None, K_new_a)

# ── Method B: cv2.remap (precomputed maps) ────────────────────────────────────
# alpha=0 → crop to valid pixels only  →  no black corners, tighter FOV
K_new_b, roi_b = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha=0)
mapx_b, mapy_b = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_b, (w, h), cv2.CV_32FC1)
undist_b = cv2.remap(distorted, mapx_b, mapy_b, cv2.INTER_LINEAR)

# ── Results ───────────────────────────────────────────────────────────────────
# Panel 3: lines straight, black triangles at corners (alpha=1 kept full FOV)
# Panel 4: lines straight, corners cropped away      (alpha=0 tight crop)
show_images([
    (clean,     'Original clean grid\n(no distortion)'),
    (distorted, '[TRUE MODEL] Distorted\n(barrel k1=-0.4)'),
    (undist_a,  'Method A: cv2.undistort\nalpha=1 → full FOV, black corners'),
    (undist_b,  'Method B: cv2.remap\nalpha=0 → no black corners, tight crop'),
])

print("Lines should be straight in panels 3 and 4.")
print("Both methods produce equivalent straight lines — difference is only the crop.")
print()
print(f"K_new (alpha=0): fx={K_new_b[0,0]:.1f}  cx={K_new_b[0,2]:.1f}")
print(f"Valid ROI (alpha=0): {roi_b}")
print()
print("Speed: remap is 3–5× faster (mapping precomputed once, reused every frame).")
print("→ Use undistort for single images.  Use remap for video loops.")

In [ ]:
# Alpha trade-off: crop vs field of view
#
# All three panels below use [TRUE MODEL] distortion — lines ARE straight in
# all of them. The only difference is how the corners are handled:
#
#   alpha=0 → crop to valid pixels  → no black corners  (recommended for pose)
#   alpha=1 → keep full frame size  → black triangles at corners
#
# Look at the CORNERS of each image, not the lines (lines are straight in all).

labels = {
    0.0: 'alpha=0  [TRUE MODEL]\nTight crop — no black borders\n→ use for pose estimation',
    0.5: 'alpha=0.5  [TRUE MODEL]\nPartial crop — small black triangles',
    1.0: 'alpha=1  [TRUE MODEL]\nFull FOV — black triangles at corners\n→ use when full FOV matters',
}

results = []
for alpha in [0.0, 0.5, 1.0]:
    K_new_a, roi = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (w, h), alpha)
    mapx_a, mapy_a = cv2.initUndistortRectifyMap(
        K_strong, dist_strong, None, K_new_a, (w, h), cv2.CV_32FC1)
    und = cv2.remap(distorted, mapx_a, mapy_a, cv2.INTER_LINEAR)
    results.append((und, labels[alpha]))

show_images(results, figsize_per=(5, 4))

print("Lines are straight in ALL three panels — alpha does not affect correctness.")
print("Alpha only controls whether empty corners are cropped away or kept as black.")
print()
print("Practical rule:")
print("  alpha=0 → ArUco detection, solvePnP, any pose pipeline.")
print("            Black corners confuse feature detectors.")
print("  alpha=1 → when you need the full original field of view.")

## 3. Real-Time Undistortion Loop

This is the pattern to use in all real-time robotics applications:

In [ ]:
# ── Real-time undistortion loop — reference pattern ──────────────────────────
#
# In a real script (not Jupyter) you would open a live camera:
#     cap = cv2.VideoCapture(0)
#     w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
#     h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
#
# Here we simulate it with synthetic frames so the cell runs to completion
# in a notebook. The structure — and the key rule — are identical:
#   PRECOMPUTE MAPS ONCE  →  APPLY remap EVERY FRAME
# ─────────────────────────────────────────────────────────────────────────────

import time

# ── 1. Load calibration (once, at startup) ───────────────────────────────────
K_rt    = K_strong.copy()     # in a real script: data = np.load('camera_calibration.npz')
dist_rt = dist_strong.copy()  #                   K_rt = data['K']; dist_rt = data['dist']
h_rt, w_rt = 480, 640

# ── 2. Precompute remap maps ONCE before the loop ────────────────────────────
K_new_rt, _ = cv2.getOptimalNewCameraMatrix(K_rt, dist_rt, (w_rt, h_rt), alpha=0)
mapx_rt, mapy_rt = cv2.initUndistortRectifyMap(
    K_rt, dist_rt, None, K_new_rt, (w_rt, h_rt), cv2.CV_32FC1)

print("Maps precomputed — K_new_rt ready for downstream pose estimation")
print(f"  K_new fx={K_new_rt[0,0]:.1f}  fy={K_new_rt[1,1]:.1f}  "
      f"cx={K_new_rt[0,2]:.1f}  cy={K_new_rt[1,2]:.1f}")

# ── 3. Per-frame loop ─────────────────────────────────────────────────────────
# In a real script:  while cap.isOpened(): ret, frame = cap.read()
# Here: 5 synthetic distorted frames to show the pattern runs at full speed.

N_FRAMES = 5
t0 = time.perf_counter()
for i in range(N_FRAMES):
    # Simulate what cap.read() returns: a distorted frame from the camera
    raw_frame  = cv2.remap(create_grid_image(h_rt, w_rt, 40),
                           mapx_rt, mapy_rt, cv2.INTER_LINEAR)

    # ← This one line is all the per-frame work:
    undistorted = cv2.remap(raw_frame, mapx_rt, mapy_rt, cv2.INTER_LINEAR)

    # All downstream work (ArUco detect, solvePnP, …) uses:
    #   frame    = undistorted   (distortion already removed)
    #   K        = K_new_rt      (updated intrinsics)
    #   dist     = np.zeros(5)   (no remaining distortion)

elapsed_ms = (time.perf_counter() - t0) / N_FRAMES * 1000
print(f"\nPer-frame remap: {elapsed_ms:.2f} ms  →  {1000/elapsed_ms:.0f} FPS ceiling")
print("\nKey rule: initUndistortRectifyMap BEFORE the loop, remap INSIDE the loop.")

## Running the real-time undistortion script

The script `scripts/pose/undistort_live_video.py` opens your webcam and shows a side-by-side comparison of the raw (distorted) frame and the corrected (undistorted) frame in real time. It precomputes the correction maps once at startup so the per-frame remapping is very fast.

---

### Before you run

You need your camera calibration file first. Run NB07 to generate `assets/calibration/camera_calibration.npz`. Without it the script uses a synthetic distortion fallback and the correction will not reflect your actual lens.

Then set up your virtual environment if you haven't already:

```bash
# 1. Create the venv (only once):
python -m venv venv

# 2. Activate it (every time you open a new terminal):
#    Windows:
venv\Scripts\activate
#    macOS / Linux:
source venv/bin/activate

# 3. Install dependencies (only once, after activating):
pip install -r requirements.txt
```

You should see `(venv)` at the start of your terminal prompt.  
If you don't, the venv is not active and the script will fail with `ModuleNotFoundError: No module named 'cv2'`.

---

### What the script does, step by step

1. Loads camera intrinsics (K, dist) from the calibration file.
2. Calls `getOptimalNewCameraMatrix` to compute a new K_new for the undistorted image — the `--alpha` parameter controls whether to crop black borders or keep the full field of view.
3. Calls `initUndistortRectifyMap` to build per-pixel remap lookup tables. This heavy step runs **once** at startup.
4. Each frame: applies `cv2.remap` using the precomputed tables — this is extremely fast and runs at full camera frame rate.
5. Labels each pane (`RAW` / `UNDISTORTED`) and overlays the current FPS.
6. In side-by-side mode, displays both half-size frames next to each other. Look at straight edges (door frames, window sills, walls) — they should appear curved in the left pane and straight in the right pane.

---

### How to run

```bash
# Default: auto-detects calibration, alpha=0 (no black borders)
python scripts/pose/undistort_live_video.py

# Specify calibration file explicitly
python scripts/pose/undistort_live_video.py --calib assets/calibration/camera_calibration.npz

# Keep full field of view (shows black borders at edges)
python scripts/pose/undistort_live_video.py --alpha 1
```

Key arguments:
- `--calib` — Path to the `.npz` calibration file from NB07.
- `--alpha` — `0` = crop to valid pixels only, no black borders (default). `1` = keep the full field of view, which shows black triangles at the corners.
- `--camera` — Camera index (default: `0`).
- `--width` / `--height` — Capture resolution (default: `1280 × 720`).

---

### Controls

| Key | Action |
|-----|--------|
| `Q` or `Esc` | Quit the live window |
| `S` | Toggle between side-by-side view and undistorted-only full-screen view |

---

### What output to expect

A window opens showing two half-size frames side by side, labelled `RAW` and `UNDISTORTED`. The FPS is shown in the top-right corner of the right pane.

At startup the terminal also prints the corrected camera matrix K_new:

```
Calibration loaded: assets/calibration/camera_calibration.npz
Camera     : 0  actual size 1280×720
alpha      : 0.0  (no black borders)
K_new      : fx=...  fy=...  cx=...  cy=...
```

No files are saved. If you use undistorted frames in a downstream pipeline (ArUco detection, solvePnP), use `K_new` as your camera matrix and pass `dist=zeros` — the distortion has already been removed by the remap step.

In [ ]:
# Benchmark: undistort vs remap performance

import time

# Create a test frame
test_frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)

# Precompute remap maps
K_new_r, _ = cv2.getOptimalNewCameraMatrix(K_strong, dist_strong, (640, 480), alpha=0)
mapx_r, mapy_r = cv2.initUndistortRectifyMap(
    K_strong, dist_strong, None, K_new_r, (640, 480), cv2.CV_32FC1)

N = 100

# Benchmark cv2.undistort
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.undistort(test_frame, K_strong, dist_strong, None, K_new_r)
t_undistort = (time.perf_counter() - t0) / N * 1000

# Benchmark cv2.remap (precomputed)
t0 = time.perf_counter()
for _ in range(N):
    _ = cv2.remap(test_frame, mapx_r, mapy_r, cv2.INTER_LINEAR)
t_remap = (time.perf_counter() - t0) / N * 1000

print(f"cv2.undistort: {t_undistort:.2f} ms/frame → {1000/t_undistort:.0f} FPS max")
print(f"cv2.remap:     {t_remap:.2f} ms/frame  → {1000/t_remap:.0f} FPS max")
print(f"Speedup: {t_undistort/t_remap:.1f}×")
print("\n→ Use remap for video — precompute once, apply every frame")

## 4. Validating Undistortion

### The straight-line test

The easiest visual validation: straight physical lines (walls, doors, table edges) should appear straight after undistortion. If they're still curved, recalibrate.

In [ ]:
# Validation: distort → undistort round-trip using your actual calibration.
#
# [TRUE MODEL] is used throughout — lines must come back perfectly straight.
# A 3× magnified version is shown so the distortion effect is visually obvious
# even when default / mild calibration values are used.

K_val, dist_val = K.copy(), dist.copy()

# ── Clean reference image ─────────────────────────────────────────────────────
orig = create_grid_image(480, 640, spacing=40)
cv2.line(orig, (0, 240), (640, 240), (0, 0, 200), 2)
cv2.line(orig, (320, 0), (320, 480), (0, 0, 200), 2)
cv2.putText(orig, 'These lines must be straight after undistort',
            (10, 460), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 200), 1)
h_v, w_v = orig.shape[:2]

# ── [TRUE MODEL] Real distortion (your actual K / dist) ───────────────────────
distorted_real = make_distorted_image(orig, K_val, dist_val)

# ── [TRUE MODEL] Magnified distortion (3× coefficients) ───────────────────────
# Makes the visual effect obvious even when real dist values are mild
dist_mag = dist_val * 3
distorted_mag = make_distorted_image(orig, K_val, dist_mag)

# ── Round-trip: undistort the real distorted image ────────────────────────────
# Lines WILL come back perfectly straight — [TRUE MODEL] guarantees this.
mapx_v, mapy_v = cv2.initUndistortRectifyMap(
    K_val, dist_val, None, K_val, (w_v, h_v), cv2.CV_32FC1)
recovered = cv2.remap(distorted_real, mapx_v, mapy_v, cv2.INTER_LINEAR)

# ── Display ───────────────────────────────────────────────────────────────────
src       = CALIB_SOURCE.upper()
d_str     = str(dist_val.flatten().round(3))
d_mag_str = str(dist_mag.flatten().round(3))

show_images([
    (orig,           'Original clean grid'),
    (distorted_real, f'[TRUE MODEL] Distorted [{src}]\ndist={d_str}'),
    (distorted_mag,  f'[TRUE MODEL] Distorted ×3 magnified\ndist={d_mag_str}'),
    (recovered,      f'Round-trip recovered [{src}]\nlines straight → undistortion OK'),
], figsize_per=(4, 4))

print(f"Calibration source : {CALIB_SOURCE}")
print(f"dist               : {dist_val.flatten()}")
print(f"dist ×3 (magnified): {dist_mag.flatten()}")
print()
print("[TRUE MODEL] used throughout — round-trip cancels exactly.")
print("Panel 2: real distortion (may look mild with default data).")
print("Panel 3: 3× magnified so the visual effect is always obvious.")
print("Panel 4: lines straight → undistortion is working correctly.")

## Recap

| Concept | Key point |
|---|---|
| Radial distortion | $k_1, k_2, k_3$ — barrel (k1<0) or pincushion (k1>0) |
| Tangential distortion | $p_1, p_2$ — lens/sensor misalignment; usually small |
| dist vector | `[k1, k2, p1, p2, k3]` — OpenCV format |
| `undistort` | Simple; recomputes map every call; use for single images |
| `remap` | Fast; precompute with `initUndistortRectifyMap`; use for video |
| alpha=0 | Crop to valid pixels only (no black borders) |
| alpha=1 | Keep full FOV, include black border areas |
| After undistortion | Use `K_new` (not original K) and `dist=zeros` for pose estimation |

---

**Next:** `09_solvePnP_explained.ipynb` — now that we have K and can remove distortion, use corresponding 3D-2D point pairs to estimate the full 6D pose.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Effect of each distortion coefficient
# ============================================================
# Create a grid image and apply distortion with ONLY ONE coefficient nonzero:
#   1. k1 only (try -0.5 and +0.5)
#   2. k2 only (try +0.2)
#   3. p1 only (try +0.1)
#   4. p2 only (try +0.1)
# Display all 5 images and describe what each coefficient does.

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# grid = create_grid_image(400, 400, spacing=40)
# K_ex = np.array([[350, 0, 200], [0, 350, 200], [0, 0, 1]], dtype=np.float64)
# cases = []
# for name, d_arr in [
#     ('Original',    [0, 0, 0, 0, 0]),
#     ('k1=-0.5',     [-0.5, 0, 0, 0, 0]),
#     ('k1=+0.5',     [0.5, 0, 0, 0, 0]),
#     ('k2=+0.2',     [0, 0.2, 0, 0, 0]),
#     ('p1=+0.15',    [0, 0, 0.15, 0, 0]),
#     ('p2=+0.15',    [0, 0, 0, 0.15, 0]),
# ]:
#     d = np.array(d_arr)
#     if np.all(d == 0):
#         cases.append((grid, name))
#     else:
#         m1, m2 = cv2.initUndistortRectifyMap(K_ex, d, None, K_ex, (400,400), cv2.CV_32FC1)
#         cases.append((cv2.remap(grid, m1, m2, cv2.INTER_LINEAR), name))
#
# show_images(cases, figsize_per=(3, 3))
# # k1<0 → barrel (lines curve outward)
# # k1>0 → pincushion (lines curve inward)
# # k2 → adds higher-order radial correction
# # p1/p2 → diagonal shift (lines shift asymmetrically)

In [ ]:
# ============================================================
# EXERCISE 2: Build a complete undistortion function
# ============================================================
# Write undistort_frame(frame, K, dist, alpha=0) that:
#   1. Computes K_new with getOptimalNewCameraMatrix
#   2. Precomputes maps with initUndistortRectifyMap
#   3. Applies remap
#   4. Returns (undistorted_frame, K_new)
#
# Note: in a real video loop, precompute maps OUTSIDE the loop.
# This exercise is for the single-frame case.

# YOUR CODE HERE
def undistort_frame(frame, K, dist, alpha=0):
    pass


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def undistort_frame(frame, K, dist, alpha=0):
#     h, w = frame.shape[:2]
#     K_new, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), alpha)
#     mapx, mapy = cv2.initUndistortRectifyMap(K, dist, None, K_new, (w, h), cv2.CV_32FC1)
#     undistorted = cv2.remap(frame, mapx, mapy, cv2.INTER_LINEAR)
#     return undistorted, K_new
#
# # Test
# test_frame = distorted_val.copy()
# result, K_new_out = undistort_frame(test_frame, K_val, dist_val, alpha=0)
# print(f"Output shape: {result.shape}")
# print(f"K_new: fx={K_new_out[0,0]:.1f}, cx={K_new_out[0,2]:.1f}")
# show_images([(test_frame, 'Distorted'), (result, 'Undistorted')])